# MCSL-069 — Question 32

**Section:** 8.2 Machine Learning, Session 9, Exercise a

**Question:** Build a multilayer feedforward network for handwritten digits.

The accompanying CSV file is a small, reproducible input dataset. Run the code cell below to see the implementation and its results.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
torch.manual_seed(69)
data = pd.read_csv("a_digits.csv")
small_images = data.drop(columns="digit").to_numpy(dtype=np.float32).reshape(-1, 8, 8) / 16.0
# Resize the bundled 8x8 handwritten digits to the requested 28x28 input shape.
X = np.stack([np.pad(np.kron(image, np.ones((3, 3))), ((2, 2), (2, 2))).ravel()
              for image in small_images]).astype(np.float32)
y = data.digit.to_numpy(dtype=np.int64)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=69, stratify=y)
X_train, X_test = torch.tensor(X_train), torch.tensor(X_test)
y_train_t = torch.tensor(y_train)
model = nn.Sequential(nn.Linear(784, 128), nn.Sigmoid(),
                      nn.Linear(128, 64), nn.Sigmoid(), nn.Linear(64, 10))
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()
for epoch in range(20):
    optimizer.zero_grad()
    loss = loss_fn(model(X_train), y_train_t)
    loss.backward()
    optimizer.step()
pred = model(X_test).argmax(dim=1).numpy()
print("Offline dataset: scikit-learn handwritten digits resized from 8x8 to 28x28 and bundled as CSV.")
print("Accuracy:", round(accuracy_score(y_test, pred), 3))
print("Confusion matrix:\n", confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, zero_division=0))


Offline dataset: scikit-learn handwritten digits resized from 8x8 to 28x28 and bundled as CSV.
Accuracy: 0.902
Confusion matrix:
 [[45  0  0  0  0  0  0  0  0  0]
 [ 0 42  2  0  1  0  0  0  1  0]
 [ 0  0 43  1  0  0  0  0  0  0]
 [ 0  0  2 37  0  1  0  2  3  1]
 [ 0  0  0  0 42  0  0  2  1  0]
 [ 0  0  0  1  0 44  0  0  0  1]
 [ 0  0  0  0  0  0 45  0  0  0]
 [ 0  0  0  0  0  0  0 44  1  0]
 [ 0  6  0  0  0  2  1  1 32  1]
 [ 0  1  0  5  1  2  0  2  2 32]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        45
           1       0.86      0.91      0.88        46
           2       0.91      0.98      0.95        44
           3       0.84      0.80      0.82        46
           4       0.95      0.93      0.94        45
           5       0.90      0.96      0.93        46
           6       0.98      1.00      0.99        45
           7       0.86      0.98      0.92        45
           8       0.80      0.74      0.77        4